# TP2/TP3 Longitudinal Analysis - Graphs

This file contains the scatterplot and model fit for the TP2/TP3 longitudinal analysis of canonical proportion (CP) as a predictor of the following speech language measures:
- GFTA
- EVT
- CTOPP
- SAILS

The weighted, scaled models are used on the scaled outcome measures.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import to_rgba
from sklearn.preprocessing import StandardScaler

# === Load TP2/TP3 file with scaled columns ===
df = pd.read_csv("tp2_tp3_master_with_scaled.csv")

# --- Weights: use existing num_clips column (no recompute) ---
if "num_clips" not in df.columns:
    raise KeyError("Expected 'num_clips' column in tp2_tp3_master_with_scaled.csv")
df["num_clips"] = pd.to_numeric(df["num_clips"], errors="coerce")
w = df["num_clips"].fillna(df["num_clips"].median())  # fallback if a few NaNs
w = w.clip(lower=1).astype(float)
w = w / w.mean()

# --- Gender dummy: 1=Female, 0=Male ---
def to_gender_dummy(x):
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)
    except Exception:
        pass
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df.get("gender", np.nan).apply(to_gender_dummy)

# --- Ensure numeric & scale covariates (age, maternal ed) ---
for col in ["age_mos", "Maternal_Education_Level", "canonical_proportion_scaled"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

sc_age = StandardScaler()
sc_med = StandardScaler()
df["age_mos_scaled"] = sc_age.fit_transform(df[["age_mos"]])
df["Maternal_Education_Level_scaled"] = sc_med.fit_transform(df[["Maternal_Education_Level"]])

# --- Helper: WLS with controls (outcome_z ~ CP_z + age_z + gender + med_z) ---
def fit_weighted_with_controls(y_col, data, weights):
    predictors = [
        "canonical_proportion_scaled",
        "age_mos_scaled",
        "gender_dummy",
        "Maternal_Education_Level_scaled",
    ]
    needed = [y_col, "num_clips"] + predictors
    keep = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if keep.empty:
        return None

    X = sm.add_constant(keep[predictors], has_constant="add")
    y = keep[y_col]
    ww = weights.loc[keep.index].to_numpy()

    model = sm.WLS(y, X, weights=ww).fit()

    # Controls held at means for plotting
    ctrl_means = {
        "age_mos_scaled": float(keep["age_mos_scaled"].mean()),
        "gender_dummy": float(keep["gender_dummy"].mean()),
        "Maternal_Education_Level_scaled": float(keep["Maternal_Education_Level_scaled"].mean()),
    }
    return {"model": model, "keep": keep, "ctrl_means": ctrl_means, "y_col": y_col}

# --- Outcomes to plot: (column, tag, y-axis label, color) ---
outcomes = [
    ("GFTA_Standard_scaled",               "A", "GFTA-2 (SS)",          "#1f77b4"),  # blue
    ("EVT_Standard_scaled",                "B", "EVT-2 (SS)",           "#ffbf00"),  # amber
    ("CTOPP_Elision_Scaled_scaled",        "C", "CTOPP-2 (SS)",         "#2ca02c"),  # green
    ("SAILS_ProportionTestCorrect_scaled", "D", "SAILS Accuracy",       "#e377c2"),  # pink
]

# --- Fit all models ---
fits = [fit_weighted_with_controls(col, df, w) if col in df.columns else None
        for col, *_ in outcomes]

# --- One GLOBAL CP grid & x-limits (every panel spans full width) ---
cp_all = df["canonical_proportion_scaled"].dropna()
x_lo, x_hi = float(cp_all.min()), float(cp_all.max())
xlim = (x_lo, x_hi)
cp_grid = np.linspace(xlim[0], xlim[1], 400)

def make_pred_X(model, ctrl_means):
    """Build X matrix for prediction on cp_grid with controls at means, matching model exog."""
    exog_names = model.model.exog_names  # e.g., ['const','canonical_proportion_scaled','age_mos_scaled','gender_dummy','Maternal_Education_Level_scaled']
    Xp = pd.DataFrame(index=range(len(cp_grid)), columns=exog_names, dtype=float)
    if "const" in exog_names:
        Xp["const"] = 1.0
    if "canonical_proportion_scaled" in exog_names:
        Xp["canonical_proportion_scaled"] = cp_grid
    for k, v in ctrl_means.items():
        if k in exog_names:
            Xp[k] = v
    # any other columns (shouldn't happen) fill with 0
    for c in exog_names:
        if Xp[c].isna().any():
            Xp[c].fillna(0.0, inplace=True)
    return Xp

# --- Figure with tight spacing + skinny legend column (same format) ---
plt.rcParams.update({
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

fig = plt.figure(figsize=(13.2, 7.6), constrained_layout=False)
gs = GridSpec(
    nrows=2, ncols=3, figure=fig,
    width_ratios=[1, 1, 0.42],   # narrow legend column
    height_ratios=[1, 1],
    wspace=0.08,
    hspace=0.25
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
]
ax_leg = fig.add_subplot(gs[:, 2]); ax_leg.axis("off")

# Map ylabels to columns
ylabel_to_col = {ylab: col for col, _, ylab, _ in outcomes}

def plot_panel(ax, fit_obj, color, panel_tag, ylabel):
    if fit_obj is None:
        ax.set_visible(False); return
    model = fit_obj["model"]
    keep  = fit_obj["keep"]
    y_col = fit_obj["y_col"]
    ctrl_means = fit_obj["ctrl_means"]

    # Bubble size & opacity by clips
    sizes = np.clip(keep["num_clips"]/40.0, 20, 600)
    cmin, cmax = keep["num_clips"].min(), keep["num_clips"].max()
    alphas = 0.35 + 0.65 * (keep["num_clips"] - cmin) / (cmax - cmin + 1e-9)
    r, g, b, _ = to_rgba(color)
    facecols = [(r, g, b, float(a)) for a in alphas]

    # Scatter
    ax.scatter(keep["canonical_proportion_scaled"], keep[y_col],
               s=sizes, edgecolor="k", linewidth=0.35, c=facecols)

    # Predictions on GLOBAL CP grid with controls held at means
    Xp = make_pred_X(model, ctrl_means)
    pred = model.get_prediction(Xp).summary_frame(alpha=0.05)
    ax.plot(cp_grid, pred["mean"], color=color, linewidth=2.6)
    ax.fill_between(cp_grid, pred["mean_ci_lower"], pred["mean_ci_upper"],
                    color=color, alpha=0.25)

    # Labels & annotations
    ax.set_xlabel("Canonical Proportion (Scaled)")
    ax.set_ylabel(ylabel)
    ax.text(0.03, 0.92, f"({panel_tag})", transform=ax.transAxes,
            fontsize=16, fontweight="bold")
    ax.text(0.05, 0.06, f"β = {model.params['canonical_proportion_scaled']:.2f}",
            transform=ax.transAxes, fontsize=18, fontweight="bold")

    # Limits & square shape
    ax.set_xlim(*xlim)
    ax.margins(x=0)
    q1, q99 = keep[y_col].quantile([0.01, 0.99])
    y_pad = 0.10 * (q99 - q1 + 1e-9)
    ax.set_ylim(float(q1 - y_pad), float(q99 + y_pad))
    ax.set_box_aspect(1)

    # Tight label padding
    ax.xaxis.labelpad = 6
    ax.yaxis.labelpad = 6

    # Full box borders
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(1)

# --- Draw panels ---
for ax, fit_obj, (col, tag, ylab, color) in zip(axes, fits, outcomes):
    plot_panel(ax, fit_obj, color, tag, ylab)

# --- Legend (clip sizes) in skinny right column ---
clip_sizes = [50, 200, 1000, 3000, 5000, 8000]
handles = [ax_leg.scatter([], [], s=np.clip(s/40.0, 20, 600), color="gray",
                          alpha=0.7, edgecolor="k", linewidth=0.35)
           for s in clip_sizes]
labels = [f"= {s} clips" for s in clip_sizes]
ax_leg.legend(handles, labels, loc="center left", frameon=True,
              borderpad=1.6, labelspacing=1.6, handletextpad=1.2,
              borderaxespad=0.6, fontsize=12, title="Clip Size",
              title_fontsize=12)

# Trim outer margins
fig.subplots_adjust(left=0.07, right=0.87, top=0.99, bottom=0.01)
plt.show()
